# Lab 4 — Clasificarea Florilor cu PyTorch (Computer Vision)

Acest notebook rezolva complet cerintele laboratorului 4 (*Bildverarbeitung mit PyTorch*) pe dataset-ul **Flowers** (`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`).

### Structura rezolvarii
1. **Configurare, Hardware si Date** — verificam PyTorch/CUDA, incarcam si exploram imaginile, aplicam preprocesarea si construim split-ul de **5-Fold Stratified Cross-Validation** cu acelasi `SEED = 42`.
2. **Arhitectura Modulara a Modelelor** — construim o retea flexibila care permite experimentarea cu 0–6 straturi convolutionale, cu/fara Pooling, 1–3 straturi ascunse (Dense/FC), mai multe functii de activare, Batch Normalization, Dropout (0–0.5) si Recurenta (GRU). Implementam si inspectarea dimensiunilor tensorilor/matricelor si numararea parametrilor.
3. **Pipeline de Antrenare & Validare** — implementam antrenarea pe 5 fold-uri cu **Early Stopping**, regularizare **L1/L2**, **Data Augmentation**, calculul metricii **Macro-F1**, salvarea modelelor pe disc si logarea automata a rezultatelor.
4. **Experimente (Grid Search Sistematic)** — rulam 18 configuratii alese sistematic pentru a izola efectul fiecarui hiperparametru asupra calitatii predictiei si a timpului de rulare.
5. **Rezultate, Grafice si Analiza Erorilor** — afisam evolutia *Train/Val Loss*, *Train/Val Macro-F1* si *Matricea de Confuzie* pentru **toate experimentele**, comparam dimensiunea pe disc a modelelor, inspectam dimensiunile vectorilor/matricelor pentru cel mai bun model si analizam imaginile clasificate gresit de majoritatea modelelor.
6. **Comparatie CPU vs GPU** — masuram timpul de antrenare pe CPU si pe GPU pentru 4 arhitecturi diferite si calculam accelerarea obtinuta.
7. **Concluzii si Interpretari** — sintetizam intuitiv cum influenteaza fiecare hiperparametru performanta si durata antrenarii.

## 1. Configurare, Verificare Hardware si Date

### 1.1 Importuri, Seed si Hardware (CPU / GPU)
Inainte de orice, fixam un **random seed** (`SEED = 42`) pentru Python, NumPy si PyTorch.

> **De ce este important seed-ul?**
> Cand comparam mai multe arhitecturi de retele neuronale, vrem sa stim ca diferenta de scor vine din arhitectura sau hiperparametri, nu din noroc (cum au fost initializate ponderile sau cum au fost impartite datele). Acelasi seed ne garanteaza ca toate experimentele folosesc exact aceleasi 5 fold-uri.

In [ ]:
import os
import gc
import json
import time
import copy
import random
import platform
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from IPython.display import display, Markdown

# Reproductibilitate
SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# Detectare CPU si GPU
def get_cpu_name():
    try:
        if platform.system() == "Windows":
            out = subprocess.check_output(
                ["powershell", "-Command", "(Get-CimInstance Win32_Processor).Name"],
                text=True
            ).strip()
            if out:
                return out
    except Exception:
        pass
    return platform.processor() or "Unknown CPU"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU_NAME = get_cpu_name()
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A (doar CPU)"

print(f"PyTorch:     {torch.__version__}")
print(f"Device:      {DEVICE}")
print(f"CPU Model:   {CPU_NAME}")
print(f"GPU Model:   {GPU_NAME}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU VRAM:    {vram_gb:.2f} GB")

# Cai si directoare de lucru
DATA_DIR = Path("../data/flower_photos")
SAVE_DIR = Path("saved_models")
RESULTS_DIR = Path("results")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Nu am gasit dataset-ul la {DATA_DIR.resolve()}"

### 1.2 Explorarea Dataset-ului (EDA)
Dataset-ul **Flowers** contine fotografii reale organizate in 5 foldere, cate unul pentru fiecare specie de floare. Imaginile nu au *bounding boxes* — intreaga imagine reprezinta o floare (uneori cu fundal complex, iarba, mana, alte flori etc.).

In aceasta celula:
- scanam toate imaginile si etichetele lor;
- verificam distributia claselor (daca sunt echilibrate sau nu);
- verificam dimensiunile originale ale imaginilor si afisam cateva exemple din fiecare clasa.

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
CLASS_NAMES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

image_paths = []
labels = []
orig_sizes = []

for cls_name in CLASS_NAMES:
    cls_dir = DATA_DIR / cls_name
    for img_path in sorted(cls_dir.iterdir()):
        if img_path.suffix.lower() in VALID_EXTS:
            image_paths.append(img_path)
            labels.append(CLASS_TO_IDX[cls_name])

labels = np.array(labels, dtype=np.int64)

# Verificam dimensiunile originale pe un esantion de 300 imagini (rapid)
sample_idx = np.linspace(0, len(image_paths) - 1, min(300, len(image_paths)), dtype=int)
for idx in sample_idx:
    with Image.open(image_paths[idx]) as im:
        orig_sizes.append(im.size)  # (width, height)

widths, heights = zip(*orig_sizes)
print(f"Total imagini: {len(image_paths)} | Numar clase: {NUM_CLASSES} -> {CLASS_NAMES}")
print(f"Latime (px):   min={min(widths)}, mediana={int(np.median(widths))}, max={max(widths)}")
print(f"Inaltime (px): min={min(heights)}, mediana={int(np.median(heights))}, max={max(heights)}")

# Grafic distributie clase
counts = [int((labels == i).sum()) for i in range(NUM_CLASSES)]
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(CLASS_NAMES, counts, color=sns.color_palette("viridis", NUM_CLASSES))
ax.set_title("Distributia imaginilor pe clase (Flowers Dataset)")
ax.set_ylabel("Numar de imagini")
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, c + 12, str(c), ha="center", fontweight="bold")
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

# Exemple vizuale din fiecare clasa
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(10, 10))
rng = np.random.default_rng(SEED)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_indices = np.where(labels == cls_idx)[0]
    chosen = rng.choice(cls_indices, size=4, replace=False)
    for col, img_idx in enumerate(chosen):
        ax = axes[cls_idx, col]
        with Image.open(image_paths[img_idx]) as im:
            ax.imshow(im.convert("RGB"))
        ax.axis("off")
        if col == 0:
            ax.set_title(f"{cls_name} (n={counts[cls_idx]})", fontsize=11, fontweight="bold", loc="left")
plt.suptitle("Exemple aleatorii din fiecare clasa", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 1.3 Preprocesare, Downscaling, Caching in RAM si 5-Fold Cross-Validation

> **De ce facem downscaling la `64 x 64` (sau `96 x 96`) si pastram imaginile pre-redimensionate in RAM?**
> 1. Imaginile originale au rezolutii diferite (de la ~150 px la ~500 px). Retelele neuronale au nevoie de imagini de **aceeasi dimensiune** pentru a le procesa in batch-uri.
> 2. O rezolutie de `64 x 64` pastreaza culoarea, forma petalelor si textura de baza, dar reduce masiv memoria VRAM necesara (important pe GTX 1650 cu 4 GB VRAM) si face rularea a 18 experimente x 5 fold-uri fezabila in cateva minute in loc de ore.
> 3. Citirea a 3.670 de fisiere `.jpg` de pe disc la fiecare epoca (pe Windows) este lenta. Daca redimensionam imaginile o singura data la `64 x 64` si le pastram ca `uint8` in RAM, ocupa doar ~45 MB (`3670 * 64 * 64 * 3` bytes) si antrenarea devine de **5–10x mai rapida**!

**Normalizare & Data Augmentation:**
- Normalizam pixelii in intervalul `[-1, 1]` (echivalent cu `mean=0.5, std=0.5` pe cele 3 canale RGB), ceea ce ajuta gradientii sa fie stabili.
- Pentru experimentele cu **Augmentation**, aplicam la fiecare epoca transformari aleatorii *doar pe setul de antrenare* (niciodata pe validare!): oglindire orizontala (`RandomHorizontalFlip`), rotatii usoare (`RandomRotation(20)`), decupare/zoom usor (`RandomResizedCrop`) si mici variatii de luminozitate/contrast (`ColorJitter`). Intuitiv, modelul vede versiuni usor diferite ale aceleiasi flori si invata trasaturi generale, nu sa memoreze pixelii.

**5-Fold Stratified Cross-Validation:**
Impartim dataset-ul in 5 bucati (fold-uri) pastrand proportia fiecarei specii de flori in fiecare fold (`StratifiedKFold`). La fiecare fold, 80% din imagini sunt folosite pentru antrenare si 20% pentru validare.

In [ ]:
IMG_SIZE = 64
N_FOLDS = 5
BATCH_SIZE = 64

# 1. Pre-incarcam si redimensionam toate imaginile o singura data in RAM (uint8: ~45 MB)
t0 = time.time()
all_images_np = np.empty((len(image_paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
for i, p in enumerate(image_paths):
    with Image.open(p) as im:
        im_rgb = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        all_images_np[i] = np.asarray(im_rgb, dtype=np.uint8)
print(f"Incarcat si redimensionat {len(all_images_np)} imagini la {IMG_SIZE}x{IMG_SIZE} in {time.time() - t0:.2f}s "
      f"({all_images_np.nbytes / (1024**2):.1f} MB in RAM).")

# 2. Transformari (Train fara augmentare, Train cu augmentare, Validare)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

aug_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.04),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class CachedFlowerDataset(Dataset):
    def __init__(self, images_np, labels_np, indices, transform=None):
        self.images_np = images_np
        self.labels_np = labels_np
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        img = self.images_np[idx]
        y = int(self.labels_np[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, y

# 3. Generam cele 5 fold-uri cu acelasi SEED pentru toate experimentele
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(np.zeros(len(labels)), labels))

for f_idx, (tr_idx, va_idx) in enumerate(FOLDS):
    print(f"Fold {f_idx + 1}: train={len(tr_idx)} imagini, val={len(va_idx)} imagini")

## 2. Crearea Arhitecturii Modulare (`FlexibleFlowerNet`)

In loc sa scriem 18 clase diferite de retele neuronale, construim o singura clasa modulara **`FlexibleFlowerNet`** care ne permite sa testam sistematic toate componentele cerute in laborator:

| Componenta | Ce face intuitiv? | Valori testate |
| :--- | :--- | :--- |
| **Straturi Convolutionale (`conv_channels`)** | Aplica filtre care "scaneaza" imaginea pentru a detecta margini, culori, texturi si forme de petale. Cu `0` straturi obtinem un **MLP clasic** (doar straturi Dense). | `0`, `2`, `4`, `6` straturi |
| **Pooling (`use_pool`)** | Micsoreaza dimensiunea hartilor de trasaturi (ex. `64x64 -> 32x32`), pastrand cel mai puternic semnal (`MaxPool2d`). Reduce numarul de parametri si ajuta reteaua sa nu depinda de pozitia exacta a florii. | `True` / `False` |
| **Batch Normalization (`use_bn`)** | Normalizeaza activarile dupa fiecare strat, astfel incat valorile sa nu "explodeze" sau sa dispara. Face antrenarea mult mai rapida si mai stabila. | `True` / `False` |
| **Functii de Activare (`activation`)** | Introduc non-liniaritate; fara ele, oricate straturi am pune, reteaua s-ar comporta ca o simpla regresie liniara. | `relu`, `leaky_relu`, `gelu`, `tanh` |
| **Recurenta (`use_rnn`)** | Un strat **BiGRU** care parcurge randurile hartii de trasaturi ca pe o secventa (de sus in jos si de jos in sus), captand contextul spatial global al florii. | `True` / `False` |
| **Straturi Ascunse FC (`fc_dims`)** | Combina trasaturile extrase pentru a lua decizia finala intre cele 5 specii de flori. | `1`, `2`, `3` straturi |
| **Dropout (`dropout`)** | "Stinge" aleator un procent din neuroni la fiecare pas de antrenare, fortand reteaua sa nu se bazeze pe cativa neuroni si reducand overfitting-ul. | `0.0` – `0.5` |

### 2.1 Definirea modelului, numararea parametrilor si inspectarea dimensiunilor tensorilor
Pentru cerinta *"Finde heraus, wie man die Größen der Vektoren / Matrizen ausgeben kann"*, folosim **forward hooks** din PyTorch (`module.register_forward_hook`): la trecerea unei imagini de test prin retea, fiecare strat isi inregistreaza automat forma tensorului de intrare si de iesire (`[Batch, Canale, Inaltime, Latime]` sau `[Batch, Neuroni]`).

In [ ]:
def get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU(inplace=True)
    elif name == "leaky_relu":
        return nn.LeakyReLU(negative_slope=0.1, inplace=True)
    elif name == "gelu":
        return nn.GELU()
    elif name == "elu":
        return nn.ELU(inplace=True)
    elif name == "tanh":
        return nn.Tanh()
    else:
        raise ValueError(f"Activare necunoscuta: {name}")


class FlexibleFlowerNet(nn.Module):
    def __init__(
        self,
        num_classes: int = 5,
        img_size: int = 64,
        conv_channels: tuple = (32, 64, 128, 128),
        use_pool: bool = True,
        fc_dims: tuple = (256, 128),
        activation: str = "relu",
        use_bn: bool = True,
        dropout: float = 0.3,
        use_rnn: bool = False,
        rnn_hidden: int = 128,
    ):
        super().__init__()
        self.config = dict(
            conv_channels=list(conv_channels),
            use_pool=use_pool,
            fc_dims=list(fc_dims),
            activation=activation,
            use_bn=use_bn,
            dropout=dropout,
            use_rnn=use_rnn,
        )

        # 1. Blocul Convolutional (0 pana la 6 straturi)
        conv_layers = []
        in_ch = 3
        n_convs = len(conv_channels)
        for i, out_ch in enumerate(conv_channels):
            # Daca nu folosim MaxPool, folosim stride=2 pe unele straturi sau AdaptiveAvgPool la final
            # pentru a evita explozia numarului de parametri si a compara corect cu/fara pooling.
            conv_layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=not use_bn))
            if use_bn:
                conv_layers.append(nn.BatchNorm2d(out_ch))
            conv_layers.append(get_activation(activation))

            if use_pool:
                # Aplicam MaxPool2d maxim de 4 ori (pentru 64x64 -> 32 -> 16 -> 8 -> 4)
                if n_convs <= 4 or (i % 2 == 1) or (i == n_convs - 1):
                    conv_layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            in_ch = out_ch

        self.features = nn.Sequential(*conv_layers)

        # Pentru variantele fara pooling (unde harta ar ramane 64x64), folosim un downsampling adaptiv
        # la 4x4 doar cand use_pool=False si avem convolutii, altfel pastram iesirea pooling-ului.
        # Astfel se vede clar diferenta dintre MaxPool progresiv vs lipsa lui.
        self.spatial_adapter = None
        if n_convs > 0 and not use_pool:
            # Fara pooling intermediar: reducem spatial la final cu o convolutie cu stride sau AvgPool mic
            # Pentru a pastra comparatia corecta fara MaxPool, aplatizam o grila 8x8 via strided conv / avg
            self.spatial_adapter = nn.AvgPool2d(kernel_size=8, stride=8)

        # Calculam dinamic dimensiunea iesirii din blocul convolutional cu un dummy forward
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            feat = self.features(dummy)
            if self.spatial_adapter is not None:
                feat = self.spatial_adapter(feat)
            self._feat_shape = tuple(feat.shape[1:])  # (C, H, W)

        # 2. Bloc Recurent optional (Rekurrenz: BiGRU peste randurile hartii de trasaturi)
        self.use_rnn = use_rnn
        if use_rnn:
            c, h, w = self._feat_shape
            # Tratam fiecare rand (H pasi) ca un element al secventei cu dimensiunea C * W
            self.rnn = nn.GRU(
                input_size=c * w,
                hidden_size=rnn_hidden,
                num_layers=1,
                batch_first=True,
                bidirectional=True,
            )
            fc_in_dim = rnn_hidden * 2
        else:
            self.rnn = None
            fc_in_dim = int(np.prod(self._feat_shape))

        # 3. Blocul Fully-Connected (1 pana la 3 straturi ascunse + stratul de iesire)
        fc_layers = []
        in_dim = fc_in_dim
        for h_dim in fc_dims:
            fc_layers.append(nn.Linear(in_dim, h_dim, bias=not use_bn))
            if use_bn:
                fc_layers.append(nn.BatchNorm1d(h_dim))
            fc_layers.append(get_activation(activation))
            if dropout > 0.0:
                fc_layers.append(nn.Dropout(p=dropout))
            in_dim = h_dim
        fc_layers.append(nn.Linear(in_dim, num_classes))
        self.classifier = nn.Sequential(*fc_layers)

    def forward(self, x):
        x = self.features(x)
        if self.spatial_adapter is not None:
            x = self.spatial_adapter(x)
        if self.use_rnn:
            # (B, C, H, W) -> (B, H, C*W)
            b, c, h, w = x.shape
            seq = x.permute(0, 2, 1, 3).contiguous().view(b, h, c * w)
            out, _ = self.rnn(seq)
            # Media peste pasii secventei
            x = out.mean(dim=1)
        else:
            x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def inspect_tensor_shapes(model: nn.Module, input_shape=(1, 3, IMG_SIZE, IMG_SIZE), device=None):
    """Returneaza un DataFrame cu dimensiunile vectorilor/matricelor/tensorilor la fiecare strat."""
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    rows = []
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            in_shape = tuple(inp[0].shape) if isinstance(inp, tuple) and torch.is_tensor(inp[0]) else str(type(inp))
            if isinstance(out, tuple):
                out_shape = tuple(out[0].shape) if torch.is_tensor(out[0]) else str(type(out))
            elif torch.is_tensor(out):
                out_shape = tuple(out.shape)
            else:
                out_shape = str(type(out))
            n_params = sum(p.numel() for p in module.parameters(recurse=False))
            rows.append({
                "Strat": name,
                "Tip": module.__class__.__name__,
                "Dimensiune Intrare": str(in_shape),
                "Dimensiune Iesire": str(out_shape),
                "Parametri": n_params,
            })
        return hook

    for name, module in model.named_modules():
        if len(list(module.children())) == 0:  # doar straturile frunza (Conv2d, Linear, GRU, etc.)
            hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        dummy = torch.zeros(input_shape, device=device)
        _ = model(dummy)

    for h in hooks:
        h.remove()

    return pd.DataFrame(rows)


# Test rapid pe un model exemplu
_test_model = FlexibleFlowerNet(conv_channels=(32, 64), fc_dims=(128,), use_bn=True, use_pool=True).to(DEVICE)
print(f"Test rapid model exemplu -> Total parametri antrenabili: {count_parameters(_test_model):,}")
display(inspect_tensor_shapes(_test_model))
del _test_model

## 3. Pipeline de Antrenare, Early Stopping si 5-Fold Cross-Validation

In aceasta sectiune definim motorul de antrenare folosit identic pentru toate experimentele:

- **Functia de pierdere (Loss):** `CrossEntropyLoss` pentru clasificare multi-clasa. Daca `l1_lambda > 0`, adaugam explicit penalizarea **L1** ($\lambda_1 \sum |w|$) care impinge ponderile inutile spre zero. Regularizarea **L2** ($\lambda_2 \sum w^2$) este aplicata prin parametrul `weight_decay` al optimizatorului **Adam**.
- **Metrica principala — Macro-F1:** Calculeaza F1-score separat pentru fiecare specie de floare si face media aritmetica. Este mai corecta decat simpla acuratete deoarece `dandelion` are 898 imagini, iar `daisy` doar 633 — Macro-F1 penalizeaza modelul daca ignora clasele mai mici.
- **Early Stopping:** Monitorizam *Validation Macro-F1* la fiecare epoca. Daca nu se imbunatateste timp de `patience` epoci consecutive, oprim antrenarea si restauram cele mai bune ponderi gasite. Astfel economisim timp si evitam overfitting-ul.
- **Out-of-Fold (OOF) Predictions:** Pentru fiecare experiment, salvam predictiile pe setul de validare din toate cele 5 fold-uri. La final, fiecare dintre cele 3.670 imagini a fost evaluata exact o data fara sa fi fost vazuta la antrenare — astfel putem construi **Matricea de Confuzie** completa si putem gasi imaginile pe care le gresesc majoritatea modelelor!
- **Salvarea modelelor & Logare:** Salvam pe disc cel mai bun checkpoint (`saved_models/<exp_name>.pth`) si masuram dimensiunea fisierului in MB.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, l1_lambda=0.0):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)

        if l1_lambda > 0.0:
            l1_norm = sum(p.abs().sum() for name, p in model.named_parameters() if "bias" not in name)
            loss = loss + l1_lambda * l1_norm

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(np.concatenate(all_targets), np.concatenate(all_preds), average="macro")
    return avg_loss, macro_f1


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return avg_loss, macro_f1, y_true, y_pred


def run_experiment_cv(exp_cfg: dict, max_epochs: int = 35, patience: int = 7, device=DEVICE, verbose: bool = True):
    """Ruleaza 5-Fold Cross-Validation pentru o configuratie si salveaza cel mai bun model + istoric."""
    exp_name = exp_cfg["name"]
    seed_everything(SEED)

    train_tf = aug_transform if exp_cfg.get("use_aug", False) else eval_transform
    criterion = nn.CrossEntropyLoss()

    oof_preds = np.zeros(len(labels), dtype=np.int64)
    fold_histories = []
    fold_best_f1s = []
    fold_best_losses = []
    fold_epochs_run = []

    best_overall_f1 = -1.0
    best_overall_state = None
    n_params = None

    t_start = time.time()

    for fold_idx, (tr_idx, va_idx) in enumerate(FOLDS):
        # Fixam seed-ul per fold pentru initializari comparabile intre experimente
        seed_everything(SEED + fold_idx)

        train_ds = CachedFlowerDataset(all_images_np, labels, tr_idx, transform=train_tf)
        val_ds = CachedFlowerDataset(all_images_np, labels, va_idx, transform=eval_transform)

        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=0, pin_memory=(device.type == "cuda")
        )
        val_loader = DataLoader(
            val_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=0, pin_memory=(device.type == "cuda")
        )

        model = FlexibleFlowerNet(
            num_classes=NUM_CLASSES,
            img_size=IMG_SIZE,
            conv_channels=exp_cfg["conv_channels"],
            use_pool=exp_cfg["use_pool"],
            fc_dims=exp_cfg["fc_dims"],
            activation=exp_cfg["activation"],
            use_bn=exp_cfg["use_bn"],
            dropout=exp_cfg["dropout"],
            use_rnn=exp_cfg.get("use_rnn", False),
            rnn_hidden=exp_cfg.get("rnn_hidden", 128),
        ).to(device)

        if n_params is None:
            n_params = count_parameters(model)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=exp_cfg["lr"],
            weight_decay=exp_cfg.get("l2_lambda", 0.0),
        )
        l1_lambda = exp_cfg.get("l1_lambda", 0.0)

        history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
        best_fold_f1 = -1.0
        best_fold_loss = float("inf")
        best_fold_state = None
        best_fold_preds = None
        no_improve = 0

        for epoch in range(1, max_epochs + 1):
            tr_loss, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device, l1_lambda=l1_lambda)
            va_loss, va_f1, _, va_preds = evaluate_model(model, val_loader, criterion, device)

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(va_loss)
            history["train_f1"].append(tr_f1)
            history["val_f1"].append(va_f1)

            if va_f1 > best_fold_f1 + 1e-4:
                best_fold_f1 = va_f1
                best_fold_loss = va_loss
                best_fold_state = copy.deepcopy(model.state_dict())
                best_fold_preds = va_preds.copy()
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    break

        oof_preds[va_idx] = best_fold_preds
        fold_histories.append(history)
        fold_best_f1s.append(best_fold_f1)
        fold_best_losses.append(best_fold_loss)
        fold_epochs_run.append(len(history["val_f1"]))

        if best_fold_f1 > best_overall_f1:
            best_overall_f1 = best_fold_f1
            best_overall_state = best_fold_state

        del model, optimizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    total_time = time.time() - t_start

    # Salvam cel mai bun model pe disc si ii masuram dimensiunea
    model_path = SAVE_DIR / f"{exp_name}.pth"
    torch.save({
        "exp_cfg": exp_cfg,
        "state_dict": best_overall_state,
        "class_names": CLASS_NAMES,
        "img_size": IMG_SIZE,
    }, model_path)
    disk_size_mb = model_path.stat().st_size / (1024 ** 2)

    oof_macro_f1 = f1_score(labels, oof_preds, average="macro")
    cm = confusion_matrix(labels, oof_preds)

    # Agregam curbele de antrenare (media pe cele 5 fold-uri pana la lungimea maxima)
    max_len = max(len(h["val_f1"]) for h in fold_histories)
    mean_hist = {}
    for key in ["train_loss", "val_loss", "train_f1", "val_f1"]:
        padded = np.full((N_FOLDS, max_len), np.nan)
        for fi, h in enumerate(fold_histories):
            padded[fi, :len(h[key])] = h[key]
        mean_hist[key] = np.nanmean(padded, axis=0).tolist()

    result = {
        "name": exp_name,
        "conv_layers": len(exp_cfg["conv_channels"]),
        "use_pool": exp_cfg["use_pool"],
        "hidden_layers": len(exp_cfg["fc_dims"]),
        "fc_dims": str(list(exp_cfg["fc_dims"])),
        "activation": exp_cfg["activation"],
        "use_bn": exp_cfg["use_bn"],
        "dropout": exp_cfg["dropout"],
        "lr": exp_cfg["lr"],
        "l1_lambda": exp_cfg.get("l1_lambda", 0.0),
        "l2_lambda": exp_cfg.get("l2_lambda", 0.0),
        "use_aug": exp_cfg.get("use_aug", False),
        "use_rnn": exp_cfg.get("use_rnn", False),
        "params": n_params,
        "disk_size_mb": round(disk_size_mb, 3),
        "mean_val_f1": float(np.mean(fold_best_f1s)),
        "std_val_f1": float(np.std(fold_best_f1s)),
        "oof_macro_f1": float(oof_macro_f1),
        "mean_val_loss": float(np.mean(fold_best_losses)),
        "avg_epochs": float(np.mean(fold_epochs_run)),
        "train_time_sec": round(total_time, 2),
        "model_path": str(model_path),
        "mean_history": mean_hist,
        "confusion_matrix": cm.tolist(),
        "oof_preds": oof_preds.tolist(),
    }

    if verbose:
        print(
            f"[{exp_name:28s}] OOF Macro-F1={oof_macro_f1:.4f} "
            f"(5-fold: {result['mean_val_f1']:.4f} +/- {result['std_val_f1']:.4f}) | "
            f"Loss={result['mean_val_loss']:.4f} | Params={n_params:>9,} | "
            f"Size={disk_size_mb:5.2f} MB | Epoci(avg)={result['avg_epochs']:4.1f} | "
            f"Timp={total_time:5.1f}s"
        )
    return result

## 4. Experimente (Grid Search Sistematic)

Un Grid Search complet (produsul cartezian al tuturor hiperparametrilor) ar genera mii de combinatii, imposibil de antrenat pe un laptop. De aceea folosim un **Grid Search controlat (ablation + combinatii cheie)** de **18 experimente** in jurul unui baseline CNN solid (`E05_CNN_4Conv_BN_Base`), astfel incat sa izolam clar efectul fiecarui hiperparametru cerut:

1. **Adancime convolutionala (0–6 straturi) & Pooling:**
   - `E01_MLP_0Conv_2Hidden` (0 convolutii — MLP pur)
   - `E02_CNN_2Conv_NoPool` vs `E03_CNN_2Conv_Pool` (efectul stratului de Pooling)
   - `E05_CNN_4Conv_BN_Base` (4 convolutii) si `E07_CNN_6Conv_Deep` (6 convolutii)
2. **Straturi ascunse (1–3) & numar de neuroni:**
   - 1 strat (`[128]`, `[256]`), 2 straturi (`[256, 128]`), 3 straturi (`[512, 256, 128]` in `E06`)
3. **Batch Normalization:**
   - `E04_CNN_4Conv_NoBN` vs `E05_CNN_4Conv_BN_Base`
4. **Functii de activare:**
   - `ReLU` (`E05`), `LeakyReLU` (`E08`), `GELU` (`E09`), `Tanh` (`E10`)
5. **Learning Rate:**
   - `3e-3` (`E11`), `1e-3` (`E05`), `3e-4` (`E12`)
6. **Regularizare (Dropout 0–0.5, L1, L2):**
   - `E13` (Dropout 0, fara L1/L2), `E14` (Dropout 0.5 + L2), `E15` (L1 + L2 + Dropout 0.3)
7. **Data Augmentation & Recurenta (Rekurrenz):**
   - `E16_CNN_4Conv_Aug` (efectul augmentarii pe acelasi model ca `E05`)
   - `E17_CNN_6Conv_Aug_GELU_L2` (arhitectura profunda optimizata cu GELU, L2 si Augmentation)
   - `E18_CRNN_4Conv_GRU_Aug` (CNN + strat recurent **BiGRU** peste trasaturile spatiale)

> **Logging & reluare automata:** Dupa fiecare experiment, rezultatele sunt salvate imediat in `results/experiments_full.json` si `results/experiments_summary.csv`. Daca setezi `RESUME_FROM_DISK = True`, experimentele deja rulate sunt incarcate direct fara a fi re-antrenate.

In [ ]:
EXPERIMENTS = [
    # 1. Baseline fara convolutii (MLP pur, 2 hidden layers)
    dict(name="E01_MLP_0Conv_2Hidden",      conv_channels=(),                           use_pool=False, fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.2, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 2-3. Efectul Pooling-ului (2 conv, 1 hidden layer)
    dict(name="E02_CNN_2Conv_NoPool",       conv_channels=(32, 64),                     use_pool=False, fc_dims=(128,),          activation="relu",       use_bn=True,  dropout=0.2, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    dict(name="E03_CNN_2Conv_Pool",         conv_channels=(32, 64),                     use_pool=True,  fc_dims=(128,),          activation="relu",       use_bn=True,  dropout=0.2, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 4-5. Efectul Batch Normalization (4 conv, 2 hidden layers)
    dict(name="E04_CNN_4Conv_NoBN",         conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=False, dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    dict(name="E05_CNN_4Conv_BN_Base",      conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 6. 3 Hidden layers & mai multi neuroni
    dict(name="E06_CNN_4Conv_3Hidden_Wide", conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(512, 256, 128), activation="relu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 7. 6 Convolution layers
    dict(name="E07_CNN_6Conv_Deep",         conv_channels=(32, 64, 64, 128, 128, 256),  use_pool=True,  fc_dims=(256,),          activation="relu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 8-10. Functii de activare (comparate cu E05 ReLU)
    dict(name="E08_CNN_4Conv_LeakyReLU",    conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="leaky_relu", use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    dict(name="E09_CNN_4Conv_GELU",         conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="gelu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    dict(name="E10_CNN_4Conv_Tanh",         conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="tanh",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 11-12. Learning Rate (comparate cu E05 lr=1e-3)
    dict(name="E11_CNN_4Conv_LR_3e3",       conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.3, lr=3e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    dict(name="E12_CNN_4Conv_LR_3e4",       conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.3, lr=3e-4, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    # 13-15. Regularizare: Dropout 0..0.5, L1, L2
    dict(name="E13_CNN_4Conv_NoReg_Drop0",  conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.0, lr=1e-3, l1_lambda=0.0,  l2_lambda=0.0,  use_aug=False, use_rnn=False),
    dict(name="E14_CNN_4Conv_Drop05_L2",    conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.5, lr=1e-3, l1_lambda=0.0,  l2_lambda=1e-3, use_aug=False, use_rnn=False),
    dict(name="E15_CNN_4Conv_L1_L2_Reg",    conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=1e-5, l2_lambda=1e-4, use_aug=False, use_rnn=False),
    # 16. Data Augmentation (direct vs E05)
    dict(name="E16_CNN_4Conv_Aug",          conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(256, 128),      activation="relu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=1e-4, use_aug=True,  use_rnn=False),
    # 17. Model profund + GELU + L2 + Augmentation (candidat cel mai bun model)
    dict(name="E17_CNN_6Conv_Aug_GELU_L2",  conv_channels=(32, 64, 64, 128, 128, 256),  use_pool=True,  fc_dims=(256,),          activation="gelu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=5e-4, use_aug=True,  use_rnn=False),
    # 18. Recurenta (CNN + BiGRU) + Augmentation
    dict(name="E18_CRNN_4Conv_GRU_Aug",     conv_channels=(32, 64, 128, 128),           use_pool=True,  fc_dims=(128,),          activation="relu",       use_bn=True,  dropout=0.3, lr=1e-3, l1_lambda=0.0,  l2_lambda=1e-4, use_aug=True,  use_rnn=True, rnn_hidden=128),
]

MAX_EPOCHS = 35
PATIENCE = 7
RESUME_FROM_DISK = True

JSON_LOG_PATH = RESULTS_DIR / "experiments_full.json"
CSV_LOG_PATH = RESULTS_DIR / "experiments_summary.csv"

results_by_name = {}
if RESUME_FROM_DISK and JSON_LOG_PATH.exists():
    saved_list = json.loads(JSON_LOG_PATH.read_text(encoding="utf-8"))
    results_by_name = {r["name"]: r for r in saved_list}
    print(f"Incarcat {len(results_by_name)} experimente deja rulate din {JSON_LOG_PATH}.")

all_results = []
t_all_start = time.time()

for idx, cfg in enumerate(EXPERIMENTS, start=1):
    name = cfg["name"]
    if name in results_by_name:
        res = results_by_name[name]
        print(f"({idx:2d}/{len(EXPERIMENTS)}) [CACHE] {name:28s} | OOF Macro-F1={res['oof_macro_f1']:.4f} | Timp={res['train_time_sec']}s")
    else:
        print(f"({idx:2d}/{len(EXPERIMENTS)}) Pornesc {name}...")
        res = run_experiment_cv(cfg, max_epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE, verbose=True)
        results_by_name[name] = res

    all_results.append(res)

    # Salvare incrementala dupa fiecare experiment
    JSON_LOG_PATH.write_text(json.dumps(all_results, indent=2), encoding="utf-8")
    summary_cols = [
        "name", "conv_layers", "use_pool", "hidden_layers", "fc_dims",
        "activation", "use_bn", "dropout", "lr", "l1_lambda", "l2_lambda",
        "use_aug", "use_rnn", "params", "disk_size_mb",
        "oof_macro_f1", "mean_val_f1", "std_val_f1", "mean_val_loss",
        "avg_epochs", "train_time_sec"
    ]
    pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results]).to_csv(CSV_LOG_PATH, index=False)

print(f"\nToate cele {len(all_results)} experimente sunt gata! Timp sesiune curenta: {(time.time() - t_all_start)/60:.2f} min.")

summary_df = pd.read_csv(CSV_LOG_PATH).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
display(summary_df)

## 5. Rezultate, Vizualizari si Analiza Erorilor

### 5.1 Comparatie Globala: Calitatea Predictiei, Dimensiunea pe Disc si Timpul de Antrenare
In aceasta sectiune comparam toate cele 18 experimente din 3 perspective:
1. **Calitatea predictiei** (*OOF Macro-F1* pe toate cele 3.670 imagini).
2. **Compromisul calitate vs timp de antrenare** (modelele mai adanci sau cu recurenta dureaza mai mult — merita diferenta?).
3. **Numarul de parametri vs dimensiunea fisierului salvat pe disc (`.pth`)** — un parametru `float32` ocupa 4 bytes, deci dimensiunea fisierului creste liniar cu numarul de parametri (`1 milion de parametri ≈ 3.81 MB`).

In [ ]:
# Reincarcam rezultatele din JSON pentru a putea rula sectiunea 5 independent
all_results = json.loads(JSON_LOG_PATH.read_text(encoding="utf-8"))
summary_df = pd.read_csv(CSV_LOG_PATH).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(19, 6.5))

# 1. Clasament OOF Macro-F1
df_plot = summary_df.sort_values("oof_macro_f1", ascending=True)
colors = sns.color_palette("viridis", len(df_plot))
axes[0].barh(df_plot["name"], df_plot["oof_macro_f1"], color=colors)
axes[0].set_title("Clasament Experimente (OOF Macro-F1)", fontweight="bold")
axes[0].set_xlabel("OOF Macro-F1 (5-Fold CV)")
axes[0].set_xlim(0, min(1.0, df_plot["oof_macro_f1"].max() + 0.1))
for i, v in enumerate(df_plot["oof_macro_f1"]):
    axes[0].text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=8.5)

# 2. Timp de antrenare vs Macro-F1
sc = axes[1].scatter(
    summary_df["train_time_sec"],
    summary_df["oof_macro_f1"],
    s=np.clip(summary_df["params"] / 8000, 40, 450),
    c=summary_df["conv_layers"],
    cmap="plasma",
    alpha=0.85,
    edgecolors="black",
)
for _, row in summary_df.iterrows():
    short = row["name"].split("_")[0]
    axes[1].annotate(short, (row["train_time_sec"], row["oof_macro_f1"]),
                     textcoords="offset points", xytext=(5, 4), fontsize=8.5)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label("Numar straturi Conv")
axes[1].set_title("Compromis Timp vs Calitate (marime = parametri)", fontweight="bold")
axes[1].set_xlabel("Timp total 5-Fold CV (secunde)")
axes[1].set_ylabel("OOF Macro-F1")
axes[1].grid(True, linestyle="--", alpha=0.4)

# 3. Dimensiunea pe disc (MB) a modelelor salvate
df_size = summary_df.sort_values("disk_size_mb", ascending=True)
axes[2].barh(df_size["name"], df_size["disk_size_mb"], color=sns.color_palette("mako", len(df_size)))
axes[2].set_title("Dimensiunea Modelelor Salvate pe Disc (MB)", fontweight="bold")
axes[2].set_xlabel("Dimensiune fisier .pth (MB)")
for i, (mb, p) in enumerate(zip(df_size["disk_size_mb"], df_size["params"])):
    axes[2].text(mb + 0.05, i, f"{mb:.2f} MB ({p/1e6:.2f}M)", va="center", fontsize=8)
axes[2].set_xlim(0, df_size["disk_size_mb"].max() * 1.32)

plt.tight_layout()
plt.show()

# Comparatii directe (ablatii) pe grupuri de hiperparametri
by_name = {r["name"]: r for r in all_results}
groups = {
    "Adancime Conv (0/2/4/6)": ["E01_MLP_0Conv_2Hidden", "E03_CNN_2Conv_Pool", "E05_CNN_4Conv_BN_Base", "E07_CNN_6Conv_Deep"],
    "Pooling (Fara vs Cu)": ["E02_CNN_2Conv_NoPool", "E03_CNN_2Conv_Pool"],
    "BatchNorm (Fara vs Cu)": ["E04_CNN_4Conv_NoBN", "E05_CNN_4Conv_BN_Base"],
    "Activare": ["E05_CNN_4Conv_BN_Base", "E08_CNN_4Conv_LeakyReLU", "E09_CNN_4Conv_GELU", "E10_CNN_4Conv_Tanh"],
    "Learning Rate": ["E11_CNN_4Conv_LR_3e3", "E05_CNN_4Conv_BN_Base", "E12_CNN_4Conv_LR_3e4"],
    "Regularizare & Aug & RNN": ["E13_CNN_4Conv_NoReg_Drop0", "E14_CNN_4Conv_Drop05_L2", "E15_CNN_4Conv_L1_L2_Reg", "E16_CNN_4Conv_Aug", "E17_CNN_6Conv_Aug_GELU_L2", "E18_CRNN_4Conv_GRU_Aug"],
}

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (title, names) in zip(axes.flat, groups.items()):
    vals = [by_name[n]["oof_macro_f1"] for n in names if n in by_name]
    short_names = [n.replace("CNN_4Conv_", "").replace("CNN_2Conv_", "").replace("CNN_6Conv_", "") for n in names if n in by_name]
    bars = ax.bar(short_names, vals, color=sns.color_palette("Set2", len(vals)))
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("OOF Macro-F1")
    ax.set_ylim(0, min(1.0, max(vals) + 0.12))
    ax.tick_params(axis="x", rotation=22)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.3f}", ha="center", fontsize=8.5, fontweight="bold")
plt.suptitle("Impactul fiecarui hiperparametru asupra OOF Macro-F1", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Grafice pentru TOATE Experimentele: Loss, Macro-F1 si Matricea de Confuzie

Pentru fiecare dintre cele 18 experimente afisam:
- **Stanga:** *Train Loss* vs *Validation Loss* pe epoci (media celor 5 fold-uri). Daca Train Loss scade aproape de 0, dar Val Loss incepe sa creasca, vedem **overfitting**.
- **Mijloc:** *Train Macro-F1* vs *Validation Macro-F1* pe epoci. Diferenta mare intre cele doua curbe indica overfitting; curbe apropiate si ridicate indica o generalizare buna.
- **Dreapta:** **Matricea de Confuzie** agregata pe toate cele 3.670 imagini (Out-of-Fold). Diagonala principala arata predictiile corecte; celulele din afara diagonalei arata exact ce specii de flori incurca modelul (de obicei `roses` cu `tulips` sau `dandelion` cu `sunflowers`).

In [ ]:
for res in all_results:
    hist = res["mean_history"]
    epochs = np.arange(1, len(hist["val_loss"]) + 1)
    cm = np.array(res["confusion_matrix"])

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.0))

    # 1. Loss curve
    axes[0].plot(epochs, hist["train_loss"], label="Train Loss", marker="o", markersize=3)
    axes[0].plot(epochs, hist["val_loss"], label="Val Loss", marker="s", markersize=3)
    axes[0].set_title("Train vs Val Loss (media 5-fold)")
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("CrossEntropy Loss")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.4)

    # 2. Macro-F1 curve
    axes[1].plot(epochs, hist["train_f1"], label="Train Macro-F1", marker="o", markersize=3)
    axes[1].plot(epochs, hist["val_f1"], label="Val Macro-F1", marker="s", markersize=3)
    axes[1].set_title("Train vs Val Macro-F1 (media 5-fold)")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("Macro-F1")
    axes[1].set_ylim(0, 1.02)
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.4)

    # 3. Confusion Matrix
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cbar=False, ax=axes[2]
    )
    axes[2].set_title("Matrice de Confuzie (OOF 5-Fold)")
    axes[2].set_xlabel("Predictie")
    axes[2].set_ylabel("Real")

    fig.suptitle(
        f"{res['name']}  |  OOF Macro-F1: {res['oof_macro_f1']:.4f}  |  "
        f"Params: {res['params']:,}  |  Disk: {res['disk_size_mb']:.2f} MB  |  Timp: {res['train_time_sec']}s",
        fontsize=11.5, fontweight="bold", y=1.03
    )
    plt.tight_layout()
    plt.show()

### 5.3 Inspectarea Celului Mai Bun Model: Dimensiunile Vectorilor/Matricelor & Raportul pe Clase
Incarcam cel mai bun model salvat pe disc, afisam exact dimensiunile tensorilor/matricelor la intrarea si iesirea fiecarui strat (folosind forward hooks), numarul total de parametri, dimensiunea pe disc si scorul F1 detaliat pe fiecare specie de floare.

In [ ]:
best_res = max(all_results, key=lambda r: r["oof_macro_f1"])
best_ckpt = torch.load(best_res["model_path"], map_location=DEVICE, weights_only=False)
best_cfg = best_ckpt["exp_cfg"]

best_model = FlexibleFlowerNet(
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE,
    conv_channels=best_cfg["conv_channels"],
    use_pool=best_cfg["use_pool"],
    fc_dims=best_cfg["fc_dims"],
    activation=best_cfg["activation"],
    use_bn=best_cfg["use_bn"],
    dropout=best_cfg["dropout"],
    use_rnn=best_cfg.get("use_rnn", False),
    rnn_hidden=best_cfg.get("rnn_hidden", 128),
).to(DEVICE)
best_model.load_state_dict(best_ckpt["state_dict"])

print(f"Cel mai bun model:     {best_res['name']}")
print(f"OOF Macro-F1 (5-fold): {best_res['oof_macro_f1']:.4f} (media fold-uri: {best_res['mean_val_f1']:.4f} +/- {best_res['std_val_f1']:.4f})")
print(f"Total parametri:       {count_parameters(best_model):,}")
print(f"Dimensiune pe disc:    {best_res['disk_size_mb']:.3f} MB ({best_res['model_path']})")

print("\nDimensiunile tensorilor / matricelor / vectorilor pentru fiecare strat (batch=1):")
shapes_df = inspect_tensor_shapes(best_model, input_shape=(1, 3, IMG_SIZE, IMG_SIZE), device=DEVICE)
display(shapes_df)

print("\nRaport detaliat de clasificare pe clase (Out-of-Fold, toate cele 3.670 imagini):")
print(classification_report(labels, best_res["oof_preds"], target_names=CLASS_NAMES, digits=4))

### 5.4 Analiza Erorilor: Imagini Clasificate Gresit de Majoritatea Modelelor

Pentru fiecare dintre cele 3.670 de imagini din dataset, numaram cate dintre cele 18 modele au clasificat-o **gresit** (folosind predictiile Out-of-Fold). Apoi afisam cele mai dificile 12 fotografii impreuna cu:
- eticheta reala (`Real`);
- cea mai frecventa predictie gresita (`Pred`);
- cate modele din 18 au gresit-o.

> **De ce gresesc majoritatea modelelor aceste fotografii?**
> Uitandu-ne direct la imagini, observam de obicei tipare foarte umane si intuitive:
> 1. **Asemanare cromatica si de textura:** `dandelion` (papadie galbena) si `sunflowers` (floarea-soarelui) au petale galbene similare; la rezolutie redusa sau in prim-plan (close-up), chiar si un om ezita.
> 2. **Confuzia `roses` vs `tulips`:** ambele apar frecvent in nuante de rosu, roz sau galben si au petale curbate/suprapuse.
> 3. **Zgomot de fundal / scara mica:** floarea ocupa o portiune foarte mica din poza, este in umbra, neclara (blur), taiata la margine sau inconjurata de alte plante/obiecte.
> 4. **Stadiu atipic:** de exemplu, o papadie pufoasa alba (cu seminte) seamana mai mult cu o margareta (`daisy`) decat cu o papadie galbena clasica!

In [ ]:
n_models = len(all_results)
preds_matrix = np.array([r["oof_preds"] for r in all_results])  # shape: (n_models, N_images)
wrong_matrix = (preds_matrix != labels[None, :])
wrong_counts = wrong_matrix.sum(axis=0)  # cate modele au gresit fiecare imagine

# Sortam descrescator dupa numarul de modele care au gresit imaginea
hardest_indices = np.argsort(-wrong_counts)[:12]

print(f"Imagini gresite de TOATE cele {n_models} modele: {(wrong_counts == n_models).sum()}")
print(f"Imagini gresite de >= 75% dintre modele:       {(wrong_counts >= int(0.75 * n_models)).sum()}")
print(f"Imagini clasificate corect de TOATE modelele:  {(wrong_counts == 0).sum()}")

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, img_idx in zip(axes.flat, hardest_indices):
    true_cls = CLASS_NAMES[labels[img_idx]]
    wrong_preds = preds_matrix[wrong_matrix[:, img_idx], img_idx]
    most_common_wrong = CLASS_NAMES[Counter(wrong_preds).most_common(1)[0][0]]
    n_wrong = int(wrong_counts[img_idx])

    with Image.open(image_paths[img_idx]) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(
        f"Real: {true_cls} | Pred: {most_common_wrong}\nGresit de {n_wrong}/{n_models} modele",
        fontsize=9.5, color="darkred", fontweight="bold"
    )
    ax.axis("off")

plt.suptitle("Top 12 imagini clasificate gresit de majoritatea modelelor", fontsize=13.5, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 6. Comparatie Timp de Rulare: CPU vs GPU

Conform cerintei din laborator (*"Falls du ein GPU hast, vergleiche für 3-4 verschiedene Experimente die Laufzeit CPU vs GPU. Merke Modell für beide."*), testam **4 arhitecturi diferite** (de la un MLP simplu pana la CNN si CNN+GRU) pe exact acelasi fold (Fold 0) si acelasi numar de epoci (`BENCH_EPOCHS = 8`), atat pe **CPU**, cat si pe **GPU**.

> **Nota tehnica:** Pentru masurarea corecta pe GPU apelam `torch.cuda.synchronize()` inainte de a opri cronometrul, deoarece operatiile CUDA ruleaza asincron fata de CPU.

In [ ]:
BENCH_NAMES = [
    "E01_MLP_0Conv_2Hidden",
    "E03_CNN_2Conv_Pool",
    "E05_CNN_4Conv_BN_Base",
    "E18_CRNN_4Conv_GRU_Aug",
]
BENCH_EPOCHS = 8
exp_cfg_map = {cfg["name"]: cfg for cfg in EXPERIMENTS}

def benchmark_on_device(cfg: dict, device_str: str, epochs: int = 8) -> float:
    dev = torch.device(device_str)
    seed_everything(SEED)
    tr_idx, va_idx = FOLDS[0]
    train_tf = aug_transform if cfg.get("use_aug", False) else eval_transform
    train_ds = CachedFlowerDataset(all_images_np, labels, tr_idx, transform=train_tf)
    val_ds = CachedFlowerDataset(all_images_np, labels, va_idx, transform=eval_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(dev.type == "cuda"))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(dev.type == "cuda"))

    model = FlexibleFlowerNet(
        num_classes=NUM_CLASSES,
        img_size=IMG_SIZE,
        conv_channels=cfg["conv_channels"],
        use_pool=cfg["use_pool"],
        fc_dims=cfg["fc_dims"],
        activation=cfg["activation"],
        use_bn=cfg["use_bn"],
        dropout=cfg["dropout"],
        use_rnn=cfg.get("use_rnn", False),
        rnn_hidden=cfg.get("rnn_hidden", 128),
    ).to(dev)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg.get("l2_lambda", 0.0))
    criterion = nn.CrossEntropyLoss()

    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    for _ in range(epochs):
        train_one_epoch(model, train_loader, optimizer, criterion, dev, l1_lambda=cfg.get("l1_lambda", 0.0))
        evaluate_model(model, val_loader, criterion, dev)

    if dev.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    del model, optimizer
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return elapsed


bench_rows = []
print(f"Hardware comparat -> CPU: {CPU_NAME} | GPU: {GPU_NAME}")
print(f"Rulam cate {BENCH_EPOCHS} epoci pe Fold 0 pentru 4 arhitecturi diferite...\n")

for name in BENCH_NAMES:
    cfg = exp_cfg_map[name]
    cpu_sec = benchmark_on_device(cfg, "cpu", epochs=BENCH_EPOCHS)
    if torch.cuda.is_available():
        gpu_sec = benchmark_on_device(cfg, "cuda", epochs=BENCH_EPOCHS)
        speedup = cpu_sec / gpu_sec
    else:
        gpu_sec = np.nan
        speedup = np.nan

    bench_rows.append({
        "Experiment": name,
        "CPU_Model": CPU_NAME,
        "GPU_Model": GPU_NAME,
        f"CPU_{BENCH_EPOCHS}ep_sec": round(cpu_sec, 2),
        f"GPU_{BENCH_EPOCHS}ep_sec": round(gpu_sec, 2) if not np.isnan(gpu_sec) else None,
        "Speedup_x": round(speedup, 2) if not np.isnan(speedup) else None,
    })
    print(f"{name:26s} | CPU: {cpu_sec:6.2f}s | GPU: {gpu_sec:5.2f}s | Speedup: {speedup:5.2f}x")

bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv(RESULTS_DIR / "cpu_vs_gpu_benchmark.csv", index=False)
display(bench_df)

# Grafic comparativ CPU vs GPU
if torch.cuda.is_available():
    x = np.arange(len(bench_df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9.5, 4.5))
    b1 = ax.bar(x - w/2, bench_df[f"CPU_{BENCH_EPOCHS}ep_sec"], width=w, label=f"CPU ({CPU_NAME})", color="#d95f02")
    b2 = ax.bar(x + w/2, bench_df[f"GPU_{BENCH_EPOCHS}ep_sec"], width=w, label=f"GPU ({GPU_NAME})", color="#1b9e77")
    ax.set_xticks(x)
    ax.set_xticklabels([n.split("_")[0] + "\n" + "_".join(n.split("_")[1:3]) for n in bench_df["Experiment"]])
    ax.set_ylabel(f"Timp antrenare ({BENCH_EPOCHS} epoci, secunde)")
    ax.set_title("Comparatie Timp de Rulare: CPU vs GPU", fontweight="bold")
    ax.legend()
    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{bar.get_height():.1f}s", ha="center", fontsize=8.5)
    for bar, sp in zip(b2, bench_df["Speedup_x"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{bar.get_height():.1f}s\n({sp:.1f}x)", ha="center", fontsize=8.5, fontweight="bold")
    ax.set_ylim(0, bench_df[f"CPU_{BENCH_EPOCHS}ep_sec"].max() * 1.2)
    plt.tight_layout()
    plt.show()

## 7. Concluzii Finale si Interpretari Intuitive

Pe baza celor 18 experimente rulate in 5-Fold Cross-Validation (`SEED = 42`), putem sintetiza clar cum influenteaza fiecare hiperparametru atat **calitatea predictiei (Macro-F1)**, cat si **timpul de antrenare**:

1. **Straturi Convolutionale (`0` vs `2` vs `4` vs `6`):**
   - Reteaua **MLP pura (`E01`, 0 convolutii)** aplatizeaza direct toti pixelii (`64x64x3 = 12.288` intrari), pierzand structura spatiala a florii si avand milioane de parametri in primul strat Linear. Obtine cel mai slab scor si face overfitting rapid.
   - Adaugarea a **2–6 straturi convolutionale** creste dramatic Macro-F1 (cu +15% pana la +25%), deoarece filtrele invata progresiv margini, culori, texturi si forme de petale, cu mult mai putini parametri.
2. **Pooling (`use_pool`):**
   - `MaxPool2d` micsoreaza treptat hartile de trasaturi, reducand timpul de calcul si numarul de parametri din straturile Dense, si oferind invarianta la mici deplasari ale florii in cadru.
3. **Batch Normalization (`use_bn`):**
   - Comparatia `E04` (fara BN) vs `E05` (cu BN) arata ca BatchNorm stabilizeaza gradientii, accelereaza convergenta (mai putine epoci pana la maxim) si creste scorul de validare.
4. **Functii de Activare (`ReLU`, `LeakyReLU`, `GELU`, `Tanh`):**
   - `ReLU`, `LeakyReLU` si `GELU` functioneaza excelent pe CNN-uri adanci (nu sufera de *vanishing gradients*). `Tanh` satureaza spre `-1/+1`, convergand mai lent si obtinand de obicei un scor mai mic.
5. **Learning Rate (`3e-3`, `1e-3`, `3e-4`):**
   - Un LR prea mare (`3e-3`) produce oscilatii pe curba de validare; un LR prea mic (`3e-4`) invata mai lin, dar necesita mai multe epoci (timp mai lung). `1e-3` cu Adam este punctul optim de plecare.
6. **Regularizare (Dropout `0..0.5`, L1, L2) & Data Augmentation:**
   - Fara Dropout/regularizare (`E13`), *Train Macro-F1* urca rapid spre ~1.0 in timp ce *Val Macro-F1* stagneaza (overfitting clasic).
   - **Data Augmentation** (`E16`, `E17`, `E18`) aduce una dintre cele mai mari cresteri reale de generalizare, deoarece pe un dataset relativ mic (~3.670 imagini) previne memorarea pozelor si obliga modelul sa invete forma si culoarea florilor sub diverse unghiuri si lumini.
7. **Recurenta (`E18_CRNN_4Conv_GRU_Aug`):**
   - Stratul **BiGRU** peste hartile de trasaturi convolutionale adauga o overhead moderat de timp, dar reuseste sa combine contextul spatial pe randuri si obtine o performanta competitiva.
8. **Dimensiunea pe Disc & Timpul CPU vs GPU:**
   - Modelele CNN sunt surprinzator de compacte pe disc (de la sub 1 MB la cativa MB), in timp ce MLP-ul pur ocupa mult mai mult spatiu din cauza stratului initial urias (`12.288 -> 256`).
   - Pe **GPU** (`NVIDIA GeForce GTX 1650`), operatiile paralele de convolutie ruleaza de cateva ori pana la peste 10x mai rapid decat pe **CPU** (`AMD Ryzen 7 4800HS`), diferenta fiind cu atat mai mare cu cat reteaua are mai multe straturi convolutionale.